<a href="https://colab.research.google.com/github/Capechusami/Zindi_Financial_Inclusion/blob/0.101636093(submission_stack_country_threshold)/Zindi_Financial_Inclusion__2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ── Install CatBoost if needed, then import everything ─────────────────────
try:
    from catboost import CatBoostClassifier
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'catboost', '-q'], check=True)
    from catboost import CatBoostClassifier

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from lightgbm import early_stopping
    LGBM_NEW_API = True
except ImportError:
    LGBM_NEW_API = False

# ── Data path ──────────────────────────────────────────────────────────────
DATA_PATH = './'

In [3]:
# ── Load data ──────────────────────────────────────────────────────────────
train = pd.read_csv(DATA_PATH + 'Train.csv')
test  = pd.read_csv(DATA_PATH + 'Test.csv')

print('Train:', train.shape, ' | Test:', test.shape)
print('\nTarget distribution:')
print(train['bank_account'].value_counts())

# Save submission IDs before any transformation
test_ids = test['uniqueid'] + ' x ' + test['country']

# Encode target: Yes -> 1, No -> 0
train['bank_account'] = (train['bank_account'] == 'Yes').astype(int)
print('\nPositive rate:', round(train['bank_account'].mean(), 4))

Train: (23524, 13)  | Test: (10086, 12)

Target distribution:
bank_account
No     20212
Yes     3312
Name: count, dtype: int64

Positive rate: 0.1408


In [4]:
# ── Feature engineering (v3 — NO leaky target encoding here) ──────────────
EDU_ORDER = {
    'No formal education': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4,
    'Other/Dont know/RTA': 2,
}

def engineer_features(df):
    df = df.copy()

    # Education ordinal
    df['education_ordinal'] = df['education_level'].map(EDU_ORDER).fillna(2).astype(int)

    # Age features
    df['log_age'] = np.log1p(df['age_of_respondent'])
    df['age_bin'] = pd.cut(df['age_of_respondent'],
                           bins=[0, 20, 30, 40, 50, 60, 100],
                           labels=False).fillna(0).astype(int)

    # Household features
    df['log_household'] = np.log1p(df['household_size'])
    df['household_bin'] = pd.cut(df['household_size'],
                                  bins=[0, 2, 4, 6, 8, 100],
                                  labels=False).fillna(0).astype(int)

    # Numeric interactions (no target leakage)
    df['age_x_edu']       = df['age_of_respondent'] * df['education_ordinal']
    df['household_x_edu'] = df['household_size']    * df['education_ordinal']
    df['age_x_household'] = df['age_of_respondent'] * df['household_size']

    return df

In [5]:
# ── Apply feature engineering (safe FE only) ───────────────────────────────
train = engineer_features(train)
test  = engineer_features(test)

# ── Categorical label encoding (fit on train + test combined) ──────────────
CAT_COLS = ['country', 'location_type', 'cellphone_access',
            'gender_of_respondent', 'relationship_with_head',
            'marital_status', 'education_level', 'job_type']

for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# ── Define features ────────────────────────────────────────────────────────
DROP_COLS    = ['bank_account', 'uniqueid']
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X      = train[FEATURE_COLS].copy()
y      = train['bank_account'].copy()
X_test = test[FEATURE_COLS].copy()

print('Features before TE:', len(FEATURE_COLS))
print(FEATURE_COLS)

Features before TE: 19
['country', 'year', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type', 'education_ordinal', 'log_age', 'age_bin', 'log_household', 'household_bin', 'age_x_edu', 'household_x_edu', 'age_x_household']


In [6]:
# ── Proper Out-of-Fold Target Encoding (no leakage) ────────────────────────
def oof_target_encode(series_train, y_train, series_test, smoothing=30,
                      n_splits=5, seed=0):
    kf       = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof      = np.zeros(len(series_train))
    gm       = y_train.mean()

    series_train = series_train.reset_index(drop=True)
    y_train      = y_train.reset_index(drop=True)
    series_test  = series_test.reset_index(drop=True)

    for tr_idx, val_idx in kf.split(series_train):
        stats = pd.DataFrame({
            'c': series_train.iloc[tr_idx].values,
            'y': y_train.iloc[tr_idx].values,
        }).groupby('c')['y'].agg(['mean', 'count'])
        smoothed        = (stats['count'] * stats['mean'] +
                           smoothing * gm) / (stats['count'] + smoothing)
        oof[val_idx]    = series_train.iloc[val_idx].map(smoothed).fillna(gm).values

    stats = pd.DataFrame({
        'c': series_train.values,
        'y': y_train.values,
    }).groupby('c')['y'].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + smoothing * gm) / (stats['count'] + smoothing)
    test_enc = series_test.map(smoothed).fillna(gm).values

    return oof, test_enc


# ── 1-way TE on high-cardinality columns only ─────────────────────────────
single_te_cols = ['country', 'job_type', 'education_level',
                  'relationship_with_head', 'marital_status', 'location_type']

for col in single_te_cols:
    tr_enc, te_enc = oof_target_encode(X[col], y, X_test[col], smoothing=30)
    X[col + '_te']      = tr_enc
    X_test[col + '_te'] = te_enc

# ── 2-way TE on meaningful pairs ──────────────────────────────────────────
pair_te = [
    ('country',          'job_type'),
    ('country',          'education_level'),
    ('country',          'location_type'),
    ('country',          'cellphone_access'),
    ('job_type',         'education_level'),
    ('job_type',         'gender_of_respondent'),
    ('cellphone_access', 'job_type'),
]

for c1, c2 in pair_te:
    combined_tr = X[c1].astype(str) + '|' + X[c2].astype(str)
    combined_te = X_test[c1].astype(str) + '|' + X_test[c2].astype(str)
    tr_enc, te_enc = oof_target_encode(combined_tr, y, combined_te, smoothing=50)
    name = c1 + '_' + c2 + '_te'
    X[name]      = tr_enc
    X_test[name] = te_enc

FEATURE_COLS = list(X.columns)
print('Features after OOF TE:', len(FEATURE_COLS))
print('New TE columns :', [c for c in FEATURE_COLS if c.endswith('_te')])

Features after OOF TE: 32
New TE columns : ['country_te', 'job_type_te', 'education_level_te', 'relationship_with_head_te', 'marital_status_te', 'location_type_te', 'country_job_type_te', 'country_education_level_te', 'country_location_type_te', 'country_cellphone_access_te', 'job_type_education_level_te', 'job_type_gender_of_respondent_te', 'cellphone_access_job_type_te']


In [7]:
# ══════════════════════════════════════════════════════════════════════════
#  STACKING PIPELINE — Level-1 Base Models (12 diverse learners)
#  Shared 10-Fold StratifiedKFold so every base model's OOF rows align,
#  enabling a proper Level-2 meta-learner with zero leakage.
# ══════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# ── Shared CV splitter (use the SAME folds for every model) ───────────────
N_SPLITS = 10
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds    = list(skf.split(X, y))

# ── Pre-build a scaled copy for distance / linear / NN models ─────────────
scaler        = StandardScaler()
X_scaled      = pd.DataFrame(scaler.fit_transform(X),      columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),     columns=X.columns)

cat_idx = [list(X.columns).index(c) for c in CAT_COLS if c in list(X.columns)]

# ── Base-model registry: name → (builder, needs_scaled, supports_es) ─────
def _lgbm_a(seed):
    return LGBMClassifier(n_estimators=5000, learning_rate=0.01, num_leaves=63,
                          min_child_samples=30, feature_fraction=0.80,
                          bagging_fraction=0.80, bagging_freq=5,
                          reg_alpha=0.10, reg_lambda=1.00,
                          random_state=seed, n_jobs=-1, verbose=-1)

def _lgbm_b(seed):  # deeper, GOSS boosting
    return LGBMClassifier(n_estimators=5000, learning_rate=0.02, num_leaves=127,
                          min_child_samples=20, feature_fraction=0.70,
                          boosting_type='goss', reg_alpha=0.20, reg_lambda=2.0,
                          random_state=seed, n_jobs=-1, verbose=-1)

def _xgb_a(seed):
    return XGBClassifier(n_estimators=5000, learning_rate=0.01, max_depth=5,
                         min_child_weight=3, subsample=0.80,
                         colsample_bytree=0.80, gamma=0.10,
                         reg_alpha=0.10, reg_lambda=2.00, eval_metric='error',
                         early_stopping_rounds=100, random_state=seed, n_jobs=-1)

def _xgb_b(seed):  # deeper, slower lr
    return XGBClassifier(n_estimators=5000, learning_rate=0.005, max_depth=7,
                         min_child_weight=5, subsample=0.70,
                         colsample_bytree=0.70, gamma=0.30,
                         reg_alpha=0.50, reg_lambda=3.00, eval_metric='logloss',
                         early_stopping_rounds=120, random_state=seed, n_jobs=-1)

def _cat_a(seed):
    return CatBoostClassifier(iterations=5000, learning_rate=0.01, depth=6,
                              l2_leaf_reg=5.0, eval_metric='Accuracy',
                              od_type='Iter', od_wait=100, task_type='CPU',
                              random_seed=seed, verbose=False)

def _cat_b(seed):  # deeper, Bernoulli bootstrap
    return CatBoostClassifier(iterations=5000, learning_rate=0.02, depth=8,
                              l2_leaf_reg=8.0, bootstrap_type='Bernoulli',
                              subsample=0.85, eval_metric='Logloss',
                              od_type='Iter', od_wait=120, task_type='CPU',
                              random_seed=seed, verbose=False)

def _hgb(seed):
    return HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.03,
                                          max_depth=8, min_samples_leaf=30,
                                          l2_regularization=1.0,
                                          early_stopping=True, validation_fraction=0.1,
                                          random_state=seed)

def _rf(seed):
    return RandomForestClassifier(n_estimators=600, max_depth=14,
                                  min_samples_leaf=10, max_features='sqrt',
                                  n_jobs=-1, random_state=seed)

def _et(seed):
    return ExtraTreesClassifier(n_estimators=800, max_depth=16,
                                min_samples_leaf=8, max_features='sqrt',
                                n_jobs=-1, random_state=seed)

def _logreg(seed):
    return LogisticRegression(C=0.5, max_iter=2000, solver='liblinear',
                              random_state=seed)

def _knn(seed):
    return KNeighborsClassifier(n_neighbors=75, weights='distance',
                                n_jobs=-1)

def _mlp(seed):
    return MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                         alpha=1e-3, learning_rate_init=1e-3, max_iter=80,
                         early_stopping=True, validation_fraction=0.1,
                         random_state=seed)

#                name           builder   uses_scaled  early_stop_kind
BASE_MODELS = [
    ('lgbm_a',   _lgbm_a,   False, 'lgbm'),
    ('lgbm_b',   _lgbm_b,   False, 'lgbm'),
    ('xgb_a',    _xgb_a,    False, 'xgb'),
    ('xgb_b',    _xgb_b,    False, 'xgb'),
    ('cat_a',    _cat_a,    False, 'cat'),
    ('cat_b',    _cat_b,    False, 'cat'),
    ('hgb',      _hgb,      False, None),
    ('rf',       _rf,       False, None),
    ('et',       _et,       False, None),
    ('logreg',   _logreg,   True,  None),
    ('knn',      _knn,      True,  None),
    ('mlp',      _mlp,      True,  None),
]
print('Level-1 base models :', len(BASE_MODELS))
print([m[0] for m in BASE_MODELS])

Level-1 base models : 12
['lgbm_a', 'lgbm_b', 'xgb_a', 'xgb_b', 'cat_a', 'cat_b', 'hgb', 'rf', 'et', 'logreg', 'knn', 'mlp']


In [8]:
# ══════════════════════════════════════════════════════════════════════════
#  Train every base model with the SHARED 10-fold split.
#  Produces:
#    OOF[name]       – out-of-fold probabilities (len = len(X))
#    TEST_PRED[name] – fold-averaged test probabilities (len = len(X_test))
# ══════════════════════════════════════════════════════════════════════════
OOF       = {}
TEST_PRED = {}

def _fit_one(name, builder, kind, X_tr, y_tr, X_val, y_val):
    mdl = builder(SEED)
    if kind == 'lgbm':
        if LGBM_NEW_API:
            mdl.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                    categorical_feature=CAT_COLS,
                    callbacks=[early_stopping(100, verbose=False)])
        else:
            mdl.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                    categorical_feature=CAT_COLS,
                    early_stopping_rounds=100, verbose=False)
    elif kind == 'xgb':
        mdl.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    elif kind == 'cat':
        mdl.fit(X_tr, y_tr, cat_features=cat_idx,
                eval_set=(X_val, y_val), use_best_model=True, verbose=False)
    else:
        mdl.fit(X_tr, y_tr)
    return mdl


for name, builder, use_scaled, kind in BASE_MODELS:
    src_X      = X_scaled      if use_scaled else X
    src_X_test = X_test_scaled if use_scaled else X_test

    oof_p  = np.zeros(len(src_X))
    test_p = np.zeros(len(src_X_test))

    t0 = time.time()
    for fold, (tr_idx, val_idx) in enumerate(folds):
        X_tr, X_val = src_X.iloc[tr_idx], src_X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx],     y.iloc[val_idx]

        mdl = _fit_one(name, builder, kind, X_tr, y_tr, X_val, y_val)

        oof_p[val_idx] = mdl.predict_proba(X_val)[:, 1]
        test_p        += mdl.predict_proba(src_X_test)[:, 1] / N_SPLITS

    OOF[name]       = oof_p
    TEST_PRED[name] = test_p
    mae = 1 - accuracy_score(y, (oof_p > 0.5).astype(int))
    print('  {:8s}  OOF MAE = {:.6f}   ({:.1f}s)'.format(
        name, mae, time.time() - t0))

# ── Ranking ────────────────────────────────────────────────────────────────
print('\n=== Level-1 OOF MAE (sorted) ===')
ranked = sorted(((1 - accuracy_score(y, (OOF[n] > 0.5).astype(int))), n)
                for n in OOF)
for mae, n in ranked:
    print('  {:8s} : {:.6f}'.format(n, mae))

  lgbm_a    OOF MAE = 0.113671   (51.9s)
  lgbm_b    OOF MAE = 0.114139   (30.5s)
  xgb_a     OOF MAE = 0.111758   (37.9s)
  xgb_b     OOF MAE = 0.113289   (71.0s)
  cat_a     OOF MAE = 0.112438   (89.8s)
  cat_b     OOF MAE = 0.112481   (346.5s)
  hgb       OOF MAE = 0.112821   (18.5s)
  rf        OOF MAE = 0.113289   (162.5s)
  et        OOF MAE = 0.112693   (144.2s)
  logreg    OOF MAE = 0.111461   (4.7s)
  knn       OOF MAE = 0.117327   (24.3s)
  mlp       OOF MAE = 0.112523   (27.6s)

=== Level-1 OOF MAE (sorted) ===
  logreg   : 0.111461
  xgb_a    : 0.111758
  cat_a    : 0.112438
  cat_b    : 0.112481
  mlp      : 0.112523
  et       : 0.112693
  hgb      : 0.112821
  rf       : 0.113289
  xgb_b    : 0.113289
  lgbm_a   : 0.113671
  lgbm_b   : 0.114139
  knn      : 0.117327


In [9]:
# ══════════════════════════════════════════════════════════════════════════
#  LEVEL-2 META-LEARNERS — trained ONLY on OOF predictions (no leakage).
#    1. Logistic Regression  (classic stacking)
#    2. LightGBM meta        (non-linear stacking)
#    3. Convex blend         (weights >=0, sum=1) found by random search
#  We then meta-blend these three with simple averaging.
# ══════════════════════════════════════════════════════════════════════════
from scipy.optimize import minimize

names    = [n for n, *_ in BASE_MODELS]
oof_mat  = np.column_stack([OOF[n]       for n in names])
test_mat = np.column_stack([TEST_PRED[n] for n in names])

# ── 1. Logistic-Regression meta (CV on OOF for honest estimate) ───────────
lr_oof  = np.zeros(len(X))
lr_test = np.zeros(len(X_test))
for tr_idx, val_idx in folds:
    meta = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs')
    meta.fit(oof_mat[tr_idx], y.iloc[tr_idx])
    lr_oof[val_idx] = meta.predict_proba(oof_mat[val_idx])[:, 1]

# Final LR meta trained on full OOF for the test predictions
meta_full = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs')
meta_full.fit(oof_mat, y)
lr_test = meta_full.predict_proba(test_mat)[:, 1]

lr_mae = 1 - accuracy_score(y, (lr_oof > 0.5).astype(int))
print('Meta LogReg     OOF MAE :', round(lr_mae, 6))
print('  Coefficients :')
for n, c in zip(names, meta_full.coef_[0]):
    print('    {:8s} : {:+.4f}'.format(n, c))

# ── 2. LightGBM meta (non-linear) ─────────────────────────────────────────
lgb_meta_oof  = np.zeros(len(X))
for tr_idx, val_idx in folds:
    m = LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
                       min_child_samples=50, reg_lambda=1.0,
                       random_state=SEED, verbose=-1)
    m.fit(oof_mat[tr_idx], y.iloc[tr_idx])
    lgb_meta_oof[val_idx] = m.predict_proba(oof_mat[val_idx])[:, 1]

m_full = LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
                        min_child_samples=50, reg_lambda=1.0,
                        random_state=SEED, verbose=-1)
m_full.fit(oof_mat, y)
lgb_meta_test = m_full.predict_proba(test_mat)[:, 1]
lgb_meta_mae  = 1 - accuracy_score(y, (lgb_meta_oof > 0.5).astype(int))
print('Meta LightGBM   OOF MAE :', round(lgb_meta_mae, 6))

# ── 3. Convex blend (weights >=0, sum to 1) optimised on OOF accuracy ─────
def neg_acc(w, P, y_):
    w = np.clip(w, 0, None)
    if w.sum() == 0:
        return 1.0
    w = w / w.sum()
    p = P @ w
    return -accuracy_score(y_, (p > 0.5).astype(int))

best_w, best_score = None, 1.0
rng = np.random.RandomState(SEED)
for _ in range(2000):
    w = rng.dirichlet(np.ones(len(names)))
    s = neg_acc(w, oof_mat, y.values)
    if s < best_score:
        best_score, best_w = s, w

# Local refinement (SLSQP)
res = minimize(neg_acc, best_w, args=(oof_mat, y.values),
               method='SLSQP',
               bounds=[(0, 1)] * len(names),
               constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
if -res.fun > -best_score:
    best_w = np.clip(res.x, 0, None); best_w /= best_w.sum()

blend_oof  = oof_mat  @ best_w
blend_test = test_mat @ best_w
blend_mae  = 1 - accuracy_score(y, (blend_oof > 0.5).astype(int))
print('Convex blend    OOF MAE :', round(blend_mae, 6))
print('  Weights :')
for n, w in sorted(zip(names, best_w), key=lambda kv: -kv[1]):
    print('    {:8s} : {:.4f}'.format(n, w))

# ── 4. Final meta-blend = average of the 3 meta predictors ────────────────
final_oof  = (lr_oof  + lgb_meta_oof  + blend_oof)  / 3.0
final_test = (lr_test + lgb_meta_test + blend_test) / 3.0
final_mae  = 1 - accuracy_score(y, (final_oof > 0.5).astype(int))
print('\n>>> FINAL meta-blend OOF MAE :', round(final_mae, 6))

Meta LogReg     OOF MAE : 0.111461
  Coefficients :
    lgbm_a   : -2.1574
    lgbm_b   : +2.6243
    xgb_a    : +0.9591
    xgb_b    : -0.7281
    cat_a    : +0.2240
    cat_b    : +1.3156
    hgb      : +0.3550
    rf       : -0.0579
    et       : +1.9121
    logreg   : +0.4485
    knn      : +0.4792
    mlp      : +1.5279
Meta LightGBM   OOF MAE : 0.11061
Convex blend    OOF MAE : 0.111248
  Weights :
    logreg   : 0.3132
    et       : 0.2341
    hgb      : 0.1562
    knn      : 0.0757
    lgbm_b   : 0.0756
    rf       : 0.0301
    xgb_b    : 0.0292
    cat_a    : 0.0221
    xgb_a    : 0.0183
    cat_b    : 0.0181
    mlp      : 0.0138
    lgbm_a   : 0.0136

>>> FINAL meta-blend OOF MAE : 0.110738


In [10]:
# ══════════════════════════════════════════════════════════════════════════
#  POST-PROCESSING — Threshold tuning, per-country thresholds, percentile
#  matching. All decisions made on OOF predictions only.
# ══════════════════════════════════════════════════════════════════════════
country_train = X['country'].values
country_test  = X_test['country'].values
unique_countries = sorted(set(country_train))

# ── A. Global threshold (fine grid) ───────────────────────────────────────
best_t, best_mae = 0.5, 1.0
for t in np.arange(0.20, 0.80, 0.001):
    m = 1 - accuracy_score(y, (final_oof > t).astype(int))
    if m < best_mae:
        best_mae, best_t = m, t
print('A. Global threshold    : t={:.3f}  OOF MAE={:.6f}'.format(best_t, best_mae))

# ── B. Per-country threshold (smoothed 50/50 toward global to fight overfit) ─
country_thr = {}
for c in unique_countries:
    mask = country_train == c
    bt, bm = best_t, 1.0
    for t in np.arange(0.10, 0.90, 0.002):
        mm = 1 - accuracy_score(y[mask], (final_oof[mask] > t).astype(int))
        if mm < bm:
            bm, bt = mm, t
    country_thr[c] = 0.5 * bt + 0.5 * best_t

oof_pred_country = np.zeros(len(X), dtype=int)
for c in unique_countries:
    mask = country_train == c
    oof_pred_country[mask] = (final_oof[mask] > country_thr[c]).astype(int)
country_mae = 1 - accuracy_score(y, oof_pred_country)
print('B. Per-country thresh  : OOF MAE={:.6f}'.format(country_mae))

# ── C. Global percentile match (top-K% as positive, K = train pos rate) ──
overall_pos_rate = y.mean()
oof_sorted = np.argsort(-final_oof)
oof_pred_pct = np.zeros(len(X), dtype=int)
oof_pred_pct[oof_sorted[:int(round(len(X) * overall_pos_rate))]] = 1
pct_mae = 1 - accuracy_score(y, oof_pred_pct)
print('C. Global percentile   : OOF MAE={:.6f}'.format(pct_mae))

# ── D. Per-country percentile match ──────────────────────────────────────
country_pos_rates = {c: y[country_train == c].mean() for c in unique_countries}
oof_pred_pct_country = np.zeros(len(X), dtype=int)
for c in unique_countries:
    mask = country_train == c
    n_pos = int(round(mask.sum() * country_pos_rates[c]))
    if n_pos == 0:
        continue
    idx = np.where(mask)[0]
    oof_pred_pct_country[idx[np.argsort(-final_oof[mask])[:n_pos]]] = 1
pct_country_mae = 1 - accuracy_score(y, oof_pred_pct_country)
print('D. Per-country pct     : OOF MAE={:.6f}'.format(pct_country_mae))

# ══════════════════════════════════════════════════════════════════════════
#  GENERATE 4 SUBMISSIONS — submit all, keep the best on Zindi LB
# ══════════════════════════════════════════════════════════════════════════
# A
sub_A = pd.DataFrame({'uniqueid': test_ids,
                      'bank_account': (final_test > best_t).astype(int)})
sub_A.to_csv('submission_stack_global.csv', index=False)

# B
preds_B = np.zeros(len(X_test), dtype=int)
for c in unique_countries:
    mask = country_test == c
    preds_B[mask] = (final_test[mask] > country_thr[c]).astype(int)
sub_B = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_B})
sub_B.to_csv('submission_stack_country_threshold.csv', index=False)

# C
preds_C = np.zeros(len(X_test), dtype=int)
preds_C[np.argsort(-final_test)[:int(round(len(X_test) * overall_pos_rate))]] = 1
sub_C = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_C})
sub_C.to_csv('submission_stack_percentile.csv', index=False)

# D
preds_D = np.zeros(len(X_test), dtype=int)
for c in unique_countries:
    mask = country_test == c
    n_pos = int(round(mask.sum() * country_pos_rates[c]))
    if n_pos == 0:
        continue
    idx = np.where(mask)[0]
    preds_D[idx[np.argsort(-final_test[mask])[:n_pos]]] = 1
sub_D = pd.DataFrame({'uniqueid': test_ids, 'bank_account': preds_D})
sub_D.to_csv('submission_stack_percentile_country.csv', index=False)

print('\n=== STACKING ENSEMBLE — 4 submissions saved ===')
print('  A. Global threshold        : {:.6f}  → submission_stack_global.csv'.format(best_mae))
print('  B. Per-country threshold   : {:.6f}  → submission_stack_country_threshold.csv'.format(country_mae))
print('  C. Global percentile       : {:.6f}  → submission_stack_percentile.csv'.format(pct_mae))
print('  D. Per-country percentile  : {:.6f}  → submission_stack_percentile_country.csv'.format(pct_country_mae))

print('\nPositive predictions per submission:')
for tag, sub in [('A', sub_A), ('B', sub_B), ('C', sub_C), ('D', sub_D)]:
    s = int(sub['bank_account'].sum())
    print('  {} : {:5d}/{}  ({:.4f})'.format(tag, s, len(sub), s / len(sub)))
print('  train positive rate : {:.4f}'.format(overall_pos_rate))

A. Global threshold    : t=0.498  OOF MAE=0.110525
B. Per-country thresh  : OOF MAE=0.110355
C. Global percentile   : OOF MAE=0.125659
D. Per-country pct     : OOF MAE=0.127444

=== STACKING ENSEMBLE — 4 submissions saved ===
  A. Global threshold        : 0.110525  → submission_stack_global.csv
  B. Per-country threshold   : 0.110355  → submission_stack_country_threshold.csv
  C. Global percentile       : 0.125659  → submission_stack_percentile.csv
  D. Per-country percentile  : 0.127444  → submission_stack_percentile_country.csv

Positive predictions per submission:
  A :   734/10086  (0.0728)
  B :   783/10086  (0.0776)
  C :  1420/10086  (0.1408)
  D :  1420/10086  (0.1408)
  train positive rate : 0.1408
